# Deep Learning Regularization & Normalization
**Level:** Advanced | **Interactive Machine Learning Curriculum**

> L1/L2 Weight Decay, Dropout, Batch Normalization, Layer Normalization, Group Normalization, and Early Stopping.

---

### 📖 Conceptual Foundations

#### Why Regularization is Crucial in Deep Learning

Deep neural networks are heavily overparameterized—they frequently possess vastly more learnable weights than training samples (often millions or billions of parameters). Without regularization, they can achieve zero training error by memorizing random noise. **Regularization** imposes structural priors that constrain parameter complexity, promoting smooth decision surfaces that generalize to unseen data.


#### Comparison of Deep Normalization Techniques

| Technique | Normalized Dimensions | Batch Dependency | Primary Domain | Inference Behavior |
| --- | --- | --- | --- | --- |
| Batch Normalization (BN) | Normalizes across batch dimension $(N)$ for each channel $(C, H, W)$ independently. | High (requires batch size $\ge 16$ for accurate batch stats). | Convolutional Networks (ResNet, ConvNeXt, YOLO). | Uses frozen exponential running mean $\mu$ and variance $\sigma^2$ computed during training. |
| Layer Normalization (LN) | Normalizes across channels and feature dimensions $(C, H, W)$ for each single sample independently. | None ($N=1$ works identically). | Transformers (BERT, GPT, ViT), Recurrent Networks (LSTMs). | Identical calculation during training and inference. |
| Group Normalization (GN) | Divides channels into groups $(G)$ and normalizes across channels within each group and spatial dims. | None ($N=1$ works identically). | Computer vision with small batch sizes (object detection, segmentation). | Identical calculation during training and inference. |
| Instance Normalization (IN) | Normalizes across spatial dimensions $(H, W)$ for each individual channel and sample independently. | None. | Style transfer, Generative Adversarial Networks (GANs). | Normalizes away style contrast while preserving content. |


#### Batch Normalization Mathematical Equations

For a mini-batch $\mathcal{B} = \{x_1, \dots, x_m\}$:

1. **Mini-batch mean:** $\mu_\mathcal{B} = \frac{1}{m} \sum_{i=1}^m x_i$
2. **Mini-batch variance:** $\sigma_\mathcal{B}^2 = \frac{1}{m} \sum_{i=1}^m (x_i - \mu_\mathcal{B})^2$
3. **Normalize:** $\hat{x}_i = \frac{x_i - \mu_\mathcal{B}}{\sqrt{\sigma_\mathcal{B}^2 + \epsilon}}$
4. **Scale and Shift (learnable affine transform):** $y_i = \gamma \hat{x}_i + \beta$

> Note: $\gamma$ and $\beta$ are learnable parameters that allow the network to restore the original representation if identity mapping is optimal.


#### Dropout & Inverted Dropout (Srivastava et al., 2014)

Dropout randomly sets a fraction $p$ of hidden unit activations to zero at each training step. This prevents **co-adaptation** of features, forcing every neuron to learn robust, self-reliant representations.

Modern frameworks implement **Inverted Dropout** during training by scaling surviving activations by $\frac{1}{1-p}$:

$$\tilde{a} = \frac{\text{mask} \odot a}{1 - p}, \quad \text{mask}_i \sim \text{Bernoulli}(1 - p)$$

**Benefit:** Inference requires zero computation or scaling—the test pass is standard forward evaluation.



In [ ]:
# %matplotlib inline
import numpy as np

# Simulated mini-batch: Batch size N=4, Features D=5
np.random.seed(42)
X = np.random.randn(4, 5) * 3 + 2 # Mean ~2, Std ~3

def batch_norm_forward(X, gamma, beta, eps=1e-5):
    # Normalize along batch axis (axis=0)
    mu = np.mean(X, axis=0, keepdims=True)
    var = np.var(X, axis=0, keepdims=True)
    X_hat = (X - mu) / np.sqrt(var + eps)
    out = gamma * X_hat + beta
    return out, mu, var

def layer_norm_forward(X, gamma, beta, eps=1e-5):
    # Normalize along feature axis (axis=-1) for each sample independently
    mu = np.mean(X, axis=-1, keepdims=True)
    var = np.var(X, axis=-1, keepdims=True)
    X_hat = (X - mu) / np.sqrt(var + eps)
    out = gamma * X_hat + beta
    return out

gamma = np.ones((1, 5))
beta = np.zeros((1, 5))

bn_out, _, _ = batch_norm_forward(X, gamma, beta)
ln_out = layer_norm_forward(X, gamma, beta)

print('Original X column means:', np.round(np.mean(X, axis=0), 2))
print('BatchNorm column means (should be ~0):', np.round(np.mean(bn_out, axis=0), 2))
print('LayerNorm row means (should be ~0):', np.round(np.mean(ln_out, axis=-1), 2))


---

### 🎯 Hands-On Experiments & Challenges
1. **Experiment with parameters:** Modify the hyperparameters or dataset sizes in the cell above.
2. **Add visual diagnostics:** Plot loss curves or feature distributions using Matplotlib/Seaborn.
3. **Test with new data:** Try running the model on synthetic noisy data or custom inputs.


In [ ]:
# Your custom experiment code here:

